# Pose-stability MD and MM-GBSA for the AChE robust core (OpenMM, 2 GPUs)
MODE below: `smoke` runs donepezil for 20 ps on both GPUs to test the whole chain; `full` runs all nine ligands, 3 ns production, two replicas (one per GPU).

In [ ]:
MODE = "full"   # 'smoke' or 'full'
import os, subprocess, sys, threading, time, shutil, glob
def sh(cmd, **kw):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable='/bin/bash', **kw)
    if r.returncode != 0:
        raise SystemExit('FAILED (exit %d): %s' % (r.returncode, cmd))
T0 = time.time()
os.environ['MPLBACKEND'] = 'Agg'   # the notebook's inline backend is not valid for the subprocesses


In [ ]:
# 1. Linux conda environment with OpenMM (CUDA), AmberTools, GAFF2 and PDBFixer
sh('curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp bin/micromamba')
os.environ['MAMBA_ROOT_PREFIX'] = '/tmp/mm'; os.environ['CONDA_OVERRIDE_CUDA'] = '12.4'
sh('/tmp/bin/micromamba create -y -q -p /tmp/mdenv -c conda-forge python=3.11 "openmm>=8.1" openmmforcefields openff-toolkit ambertools pdbfixer rdkit pandas numpy scipy')
PY = '/tmp/mdenv/bin/python'
os.environ['PATH'] = '/tmp/mdenv/bin:' + os.environ['PATH']   # antechamber / sqm (AmberTools) must be found for AM1-BCC charges
os.environ['AMBERHOME'] = '/tmp/mdenv'
sh(PY + ' -c "import openmm; from openmm import Platform; print(openmm.__version__, [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())])"')
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')

In [ ]:
# 2. copy the inputs
base = None
for r, d, f in os.walk('/kaggle/input'):
    if 'md_run.py' in f:
        base = os.path.dirname(r); break
assert base, 'input dataset not found'
root = '/tmp/mdpkg'; shutil.copytree(base, root, dirs_exist_ok=True)
os.makedirs('/kaggle/working/md_results', exist_ok=True)
print(os.listdir(root))

In [ ]:
# 3. reduced run for the remaining quota: ligands split across the two GPUs (one replica each), 2 ns, no new ligand after 1.7 h
ngpu = int(subprocess.run('nvidia-smi -L | wc -l', shell=True, capture_output=True, text=True).stdout.strip() or 1)
ns = 0.02 if MODE == 'smoke' else 2.0
extra = ' --only donepezil_crystal_4EY7' if MODE == 'smoke' else ''
procs = []
def pump(p, tag):
    for line in p.stdout:
        print('[%s] %s' % (tag, line.rstrip()[:400]), flush=True)
for k in range(min(2, max(1, ngpu))):
    cmd = '%s %s/scripts/md_run.py --gpu %d --seed %d --ns %s --shard %d --nshards 2 --budget_h 1.7 --out /kaggle/working/md_results%s' % (PY, root, k, k + 1, ns, k, extra)
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t = threading.Thread(target=pump, args=(p, 'gpu%d' % k), daemon=True); t.start(); procs.append((p, t))
for p, t in procs:
    p.wait(); t.join()
print('exit codes', [p.returncode for p, _ in procs], 'elapsed h', round((time.time() - T0) / 3600, 2))

In [ ]:
import pandas as pd
fs = sorted(glob.glob('/kaggle/working/md_results/md_summary_seed*.csv'))
assert fs, 'no MD summary was written'
df = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
print(df.drop(columns=[c for c in df.columns if c.startswith('polar')], errors='ignore').to_string())
df.to_csv('/kaggle/working/md_results/MD_SUMMARY_ALL.csv', index=False)
print('NOTEBOOK_COMPLETED_OK')